In [ ]:
!pip install auto-gptq optimum accelerate transformers

In [ ]:
!nvidia-smi

In [ ]:
from auto_gptq import AutoGPTQForCausalLM
from transformers import AutoTokenizer

model_id = "TheBloke/Llama-2-7B-Chat-GPTQ"

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

model = AutoGPTQForCausalLM.from_quantized(
    model_id,
    device_map="auto",              # ✅ FIX: Don't use device="cuda"
    use_safetensors=True,
    trust_remote_code=True,
    use_triton=False                # Optional, for Colab stability
)

In [ ]:
# Define prompt
prompt = "Explain the difference between GPTQ and AWQ in simple terms."

# Tokenize prompt
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

In [ ]:
# Generate output
outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=True,
    top_p=0.95,
    temperature=0.7
)

In [ ]:
# Decode output
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print(response)

In [ ]:
# text_gen = pipeline("text-generation", model=model, tokenizer=tokenizer)
# output = text_gen(prompt, max_new_tokens=256, do_sample=True)
# print(output[0]['generated_text'])

In [ ]:
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
from transformers import AutoTokenizer

model_id = "tiiuae/falcon-rw-1b"

quant_config = BaseQuantizeConfig(
    bits=4,
    group_size=128,
    desc_act=False
)

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

# Fix padding token issue
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoGPTQForCausalLM.from_pretrained(
    model_id,
    quantize_config=quant_config,
    trust_remote_code=True
)

# Calibration texts
example_texts = [
    "What is AI?",
    "Explain the purpose of Falcon model.",
    "List 3 benefits of GPTQ.",
    "How does quantization work in LLMs?",
    "Give an example of a large language model.",
]

# Tokenize
examples = tokenizer(example_texts, return_tensors="pt", padding=True, truncation=True)
examples = [ {k: v[i] for k, v in examples.items()} for i in range(len(example_texts)) ]

In [ ]:
output_dir = "falcon-rw-1b-gptq"
model.save_quantized(output_dir, use_safetensors=True)
tokenizer.save_pretrained(output_dir)
# IMPORTANT: save config too
from transformers import AutoConfig
AutoConfig.from_pretrained(model_id, trust_remote_code=True).save_pretrained(output_dir)

In [ ]:
!pip install -U "transformers>=4.37" accelerate
!pip install -U pip setuptools wheel ninja
!pip install -U "torch" "torchvision" --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers accelerate
!pip install "auto-gptq>=0.7.1" --no-build-isolation
# (runtime restart recommended)

In [ ]:
from auto_gptq import AutoGPTQForCausalLM
from transformers import AutoTokenizer
model_path = "falcon-rw-1b-gptq"
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
model = AutoGPTQForCausalLM.from_quantized(
    model_path,
    device_map="auto",
    trust_remote_code=False,   # ⬅️ avoids modeling_falcon.py requirement
    use_safetensors=True,
    use_triton=False,
    disable_exllamav2=True     # ⬅️ suppress the exllama warning
)

In [ ]:
import torch
import time

# Evaluation prompts
prompts = [
    "What is the capital of India?",
    "Explain how quantization improves inference.",
    "What is the Falcon model used for?",
    "List 3 applications of LLMs.",
    "Define GPTQ in simple terms.",
]

# Evaluation loop
total_time = 0.0
all_outputs = []

for i, prompt in enumerate(prompts):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Start timing
    start = time.time()

    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False  # deterministic output for eval
    )

    # End timing
    end = time.time()
    inference_time = end - start
    total_time += inference_time

    # Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    all_outputs.append((prompt, response, round(inference_time, 3)))

# Print all results
for i, (prompt, response, t) in enumerate(all_outputs):
    print(f"\n Prompt {i+1}: {prompt}")
    print(f" Response: {response}")
    print(f" Inference Time: {t} seconds")

# Average time
avg_time = total_time / len(prompts)
print(f"\n Average Inference Time: {round(avg_time, 3)} seconds per prompt")

In [ ]:
!pip -q install -U huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import whoami
whoami()

In [ ]:
from huggingface_hub import upload_folder
upload_folder(
    repo_id="sunny199/quantized-falcon-rw-1b",
    folder_path="./falcon-rw-1b-gptq",
    commit_message="Upload GPTQ quantized Falcon model"
)

In [ ]:
import os, glob, json
path = "falcon-rw-1b-gptq"
print(os.listdir(path))  # -> quantize_config.json, tokenizer files, gptq_model-4bit-*.safetensors

with open(f"{path}/quantize_config.json") as f:
    print(json.load(f))  # -> {"bits": 4, "group_size": 128, "desc_act": false, ...}
print(glob.glob(f"{path}/gptq_model-*.safetensors"))


# AWQ Quantization for LLM